In [1]:
from langchain.agents import create_agent, AgentState
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
load_dotenv()
model = ChatOllama(model='qwen2.5:3b')

In [2]:
from langgraph.checkpoint.memory import InMemorySaver
class CustomStateAgent(AgentState):
    user_id:str
    preferences:dict
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    state_schema=CustomStateAgent
)
result = agent.invoke({
    'messages':[{'role':'user', 'content':'Hello'}],
    'user_id':'user123',
    'preferences':{'theme':'dark'}
    },
    {'configurable':{'thread_id':'1'}}
    )
result2 = agent.invoke({
    'messages':[{'role':'user','content':'Who are you?'}]},
    {'configurable':{'thread_id':'2'}}
    )
final_result=agent.invoke({
    'messages':[{'role':'user','content':'What is the latest date you remember?'}]},
    {'configurable':{'thread_id':'1'}}
    )
final_result

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='b6b1c244-5718-474e-b09b-dfefbefe20d0'),
  AIMessage(content='Hello! How can I assist you today? Whether you need information on a particular topic, help with a task, or just want to chat, feel free to let me know how I can assist you.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-24T10:54:57.6653449Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11454868700, 'load_duration': 5237229600, 'prompt_eval_count': 30, 'prompt_eval_duration': 1784322300, 'eval_count': 42, 'eval_duration': 4357769000, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019befa4-6198-7e62-b707-3b09b7b2c3a8-0', usage_metadata={'input_tokens': 30, 'output_tokens': 42, 'total_tokens': 72}),
  HumanMessage(content='What is the latest date you remember?', additional_kwargs={}, response_metadata={}, id='e4950256-6a68-4c8c-98a2-0793ca0d5e67'),
  AIMessage(con

In [3]:
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_model
from langchain_core.runnables import RunnableConfig
from langgraph.runtime import Runtime
from typing import Any

@before_model
def trim_messgs(state:AgentState, runtime:Runtime)->dict[str,Any]|None:
    """Keep only last few relevant messages to fit the context window of the model."""
    messages = state['messages']
    if len(messages)<=3:
        return None
    first_message = messages[0]
    recent_messages = messages[-3:] if len(messages)%2==0 else messages[-4:]
    new_message = [first_message]+recent_messages
    return {
        'messages':[
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_message
        ]
    }
agent = create_agent(
    model=model,
    middleware=[trim_messgs],
    checkpointer=InMemorySaver()
)
config:RunnableConfig = {'configurable':{'thread_id':'1'}}
agent.invoke({'messages':'Hi I am Chandler Bing'}, config)
agent.invoke({'messages':'What date is it?'}, config)
agent.invoke({'messages':'What is your job?'}, config)
final_msg = agent.invoke({'messages':'What is my name?'}, config)
final_msg['messages'][-1].pretty_print()

================================== Ai Message ==================================

Your name is Chandler Bing. Is there anything specific you would like to know about it or discuss further?


In [ ]:
from langchain.agents.middleware import after_model
@before_model
def delete_messgs(state:AgentState, runtime:Runtime):
    """Delete all messages except first-one from the context."""
    messages = state['messages'] 
    first_msg = messages[0]
    last_msg = messages[-1]
    msg_contxt = [first_msg] + [last_msg]
    if len(messages) > 2:
        return {
            'messages' : [RemoveMessage(id=REMOVE_ALL_MESSAGES), *msg_contxt],
        }
agent = create_agent(
    model=model,
    middleware=[delete_messgs],
    checkpointer=InMemorySaver()
)
config:RunnableConfig = {'configurable':{'thread_id':'2'}}
agent.invoke({'messages':'Hi I am Joey Tribbiani'},config)
agent.invoke({'messages':'What date is it?'}, config)
agent.invoke({'messages':'What is your job?'}, config)
final_resp = agent.invoke({'messages':'What is my name?'},config)
final_resp

{'messages': [HumanMessage(content='Hi I am Joey Tribiaani', additional_kwargs={}, response_metadata={}, id='9ad65326-e70c-4b4c-bf40-f5790d41e8c5'),
  HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='e72d28b3-4d75-4a99-8e4f-add2bfe6995c'),
  AIMessage(content='Hello Joey! Your name is actually Joey Tribbiani, not Joey Tribiaanni. How can I assist you today? Are you looking for information about your character or anything else related to the show "The King of Queens"?', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-24T13:19:38.5993045Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5447348400, 'load_duration': 111885000, 'prompt_eval_count': 42, 'prompt_eval_duration': 391793300, 'eval_count': 47, 'eval_duration': 4878026600, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019bf028-ef1e-7a13-86af-399514ba39fe-0', usage_metadata={'input_tokens': 42, 'output_tokens': 47, 'total_